# 4 · Scale: Criteo uplift dataset

13.9M users, 12 anonymized features, ~85 % treated. First load downloads ~300 MB and caches a parquet file.
Causal Forest is skipped at this size; T- and X-learner with LightGBM scale linearly.

In [1]:
import time

import pandas as pd
from sklearn.model_selection import train_test_split

from uplift_causal.data import load_criteo
from uplift_causal.evaluate_uplift import budget_table, diff_means, qini_ci, qini_curve
from uplift_causal.learners import fit_predict
from uplift_causal.plots import qini_plot, save

N = 2_000_000

/Users/p.schmidt-petter/Desktop/dev/dev/uplift-causal-marketing/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_criteo(n=N)
print(df.t.mean(), diff_means(df.y, df.t))
train, test = train_test_split(df, test_size=0.3, stratify=df.t * 2 + df.y, random_state=0)
X_train, X_test = train.drop(columns=["t", "y"]), test.drop(columns=["t", "y"])

0.8503245 (np.float64(0.010383885763348324), np.float64(0.009627037472263286), np.float64(0.011140734054433363))


In [3]:
t0 = time.perf_counter()
models, tau = fit_predict(X_train, train.t, train.y, X_test, names=["T-Learner", "X-Learner"])
print(f"{time.perf_counter() - t0:.1f}s")

125.0s


In [4]:
pd.DataFrame(
    {k: qini_ci(test.y, test.t, v, n_boot=50) for k, v in tau.items()}, index=["qini", "lo", "hi"]
).T

,qini,lo,hi
T-Learner,0.002586,0.001976,0.002985
X-Learner,0.002707,0.002183,0.003115


In [5]:
save(qini_plot({k: qini_curve(test.y, test.t, v) for k, v in tau.items()}, "Qini curve · Criteo"), "qini_criteo")
budget_table(test.y, test.t, tau["X-Learner"], n_boot=50).round(3)

,cost_share,effect_share
0,0.1,0.531
1,0.2,0.701
2,0.3,0.785
3,0.5,0.861
4,0.7,0.889
5,1.0,1.000
